# Excel functions

In [60]:
import requests
import pandas as pd
from IPython.display import display
from IPython.core.display import HTML
from os import path
from bs4 import BeautifulSoup, NavigableString
from time import sleep
from pathlib import Path
from copy import deepcopy
import re

ROOT_URL = "https://support.microsoft.com"
ROOT_URI = f"{ROOT_URL}/en-us/office"
MAIN_URL = f"{ROOT_URI}/excel-functions-alphabetical-b3944572-255d-4efb-bb96-c6d90033e188"

OPTIONS = [
    "display.max_rows",
    None,
    "display.max_columns",
    "display.max_colwidth",
    None,
]
ROOT_ABSPATH = "/utkusarioglu-com/workshops/backtesting-workshop"
assets_abspath = path.join(ROOT_ABSPATH, "assets/excel")
artifacts_abspath = path.join(ROOT_ABSPATH, "artifacts/excel")
subpages_abspath = path.join(artifacts_abspath, "subpages")
source_abspath = path.join(assets_abspath, "index.html")
retrieved_abspath = path.join(artifacts_abspath, "retrieved.txt")
failed_abspath = path.join(artifacts_abspath, "failed.txt")
target_abspath = path.join(artifacts_abspath, "functions.csv")

In [10]:
# Get main file


def fetch_main_page():
    response = requests.get(MAIN_URL)
    if response.status_code != 200:
        raise RuntimeError("FETCH_FAIL")
    with open(source_abspath, "w") as f:
        f.write(response.text)


# fetch_main_page()

In [11]:
html = open(source_abspath, "r").read()
html

'\n<!DOCTYPE html>\n<html lang="en-US" dir="ltr">\n<head>\n\t<meta charset="utf-8" />\n\t<meta name="viewport" content="width=device-width, initial-scale=1.0" />\n\t<title>Excel functions (alphabetical) - Microsoft Support</title>\n\t\n\t\n\t\t<link rel="canonical" href="https://support.microsoft.com/en-us/office/excel-functions-alphabetical-b3944572-255d-4efb-bb96-c6d90033e188" />\n\t\t\n\t\t\t<link rel="alternate" hreflang="ar-SA" href="https://support.microsoft.com/ar-sa/office/%D8%AF%D8%A7%D9%84%D8%A7%D8%AA-excel-%D8%A8%D8%A7%D9%84%D8%AA%D8%B1%D8%AA%D9%8A%D8%A8-%D8%A7%D9%84%D8%A3%D8%A8%D8%AC%D8%AF%D9%8A-b3944572-255d-4efb-bb96-c6d90033e188" />\n\t\t\t<link rel="alternate" hreflang="bg-BG" href="https://support.microsoft.com/bg-bg/office/%D1%84%D1%83%D0%BD%D0%BA%D1%86%D0%B8%D0%B8-%D0%BD%D0%B0-excel-%D0%BF%D0%BE-%D0%B0%D0%B7%D0%B1%D1%83%D1%87%D0%B5%D0%BD-%D1%80%D0%B5%D0%B4-b3944572-255d-4efb-bb96-c6d90033e188" />\n\t\t\t<link rel="alternate" hreflang="cs-CZ" href="https://support.mic

In [76]:
def fetch_pages(html):
    soup = BeautifulSoup(html, "html.parser")
    table = soup.table
    anchors = table.find_all("a")
    retrieved = []
    failed = []
    print(f"Retrieving {len(anchors)} entries")
    for anchor in anchors:
        url = ROOT_URL + anchor["href"]
        filename = url.rsplit("/", 1)[-1] + ".html"
        response = requests.get(url)
        if response.status_code != 200:
            print(f"Failed: {url}", response.status_code)
            failed.append(f"{url} {response.status_code}")
            continue
        with open(path.join(subpages_abspath, filename), "w") as f:
            f.write(response.text)
        retrieved.append(url)
        print(f"Retrieved: {filename}")
        sleep(0.5)

    with open(retrieved_abspath, "w") as f:
        f.write("\n".join(retrieved))
    with open(failed_abspath, "w") as f:
        f.write("\n".join(failed))


# fetch_pages()

In [56]:
# Process soups
def replace_function_name(nav_string, title_text):
    if isinstance(nav_string, NavigableString):
        replacement = nav_string.replace(title_text, "___")
        nav_string.replace_with(replacement)


def process_soup(html):
    soup = BeautifulSoup(html, "html.parser")
    title_text = soup.h1.text.rsplit(" ", 1)[0].strip()
    title_h2_soup = BeautifulSoup(f"<h2>{title_text}</h2>", "html.parser")
    title_p_soup = BeautifulSoup(f"<p>{title_text}</p>", "html.parser")
    sections = soup.article.find_all("section")
    description_soup = None
    syntax_soup = None
    remarks_soup = None
    examples_soup = None
    for section in sections:
        if section.h2 is not None:
            if section.h2.text == "Description":
                description_soup = section
            if section.h2.text == "Syntax":
                syntax_soup = section
            if section.h2.text == "Remarks":
                remarks_soup = section
            if section.h2.text == "Examples":
                examples_soup = section
    if description_soup is None:
        description_soup = sections[0]

    # Description corrections
    if description_soup.h2 is not None:
        description_soup.h2.decompose()

    sup_arg_row = description_soup.find(class_="supARG-row")
    if sup_arg_row is not None:
        sup_arg_row.decompose()

    ocp_video = description_soup.find(class_="ocpVideo")
    if ocp_video is not None:
        ocp_video.decompose()

    anchors = description_soup.find_all("a")
    for anchor in anchors:
        slash_start = anchor["href"].startswith("/")
        hash_start = anchor["href"].startswith("#")
        if slash_start or hash_start:
            anchor["href"] = ROOT_URL + anchor["href"]

    # Summary corrections
    summary_text_raw = description_soup.find("p").text
    summary_text_arr = summary_text_raw.split(". ")
    if len(summary_text_arr) > 1:
        summary_text = summary_text_arr[0] + "."
    else:
        summary_text = summary_text_arr[0]
    sorted_title_text = [e.strip() for e in sorted(title_text.split(","))]
    sorted_title_text = sorted(sorted_title_text, key=len, reverse=True)
    for part in sorted_title_text:
        summary_text = re.sub(part, "___", summary_text)
    summary_soup = BeautifulSoup(f"<p>{summary_text}</p>", "html.parser")

    combined_description = BeautifulSoup(
        "".join(
            [
                item.prettify()
                for item in [
                    title_h2_soup,
                    description_soup,
                    syntax_soup,
                    examples_soup,
                    remarks_soup,
                ]
                if item is not None
            ]
        ),
        "html.parser",
    )

    return {
        "title_text": title_text,
        "title_h2_soup": title_h2_soup,
        "title_p_soup": title_p_soup,
        "description_soup": description_soup,
        "syntax_soup": syntax_soup,
        "remarks_soup": remarks_soup,
        "examples_soup": examples_soup,
        "summary_soup": summary_soup,
        "has_description": description_soup is not None,
        "has_syntax": syntax_soup is not None,
        "has_remarks": remarks_soup is not None,
        "has_examples": examples_soup is not None,
        "has_summary": summary_soup is not None,
        "combined_description_soup": combined_description,
    }


def soup_generator():
    files = [f for f in Path(subpages_abspath).iterdir() if f.is_file()]
    for f in files:
        with open(f, "r") as f:
            html = f.read()
            yield process_soup(html)


entries = []
counter = 0
for props in soup_generator():
    counter += 1
    # if counter > 10:
    #     break
    tags = ["MicrosoftOffice-2021"]
    if props["has_examples"]:
        tags.append("ExcelHasExamples")
    if props["has_remarks"]:
        tags.append("ExcelHasRemarks")
    if props["has_syntax"]:
        tags.append("ExcelHasSyntax")
    entries.append(
        {
            "title_p_html": props["title_p_soup"].prettify(),
            "description_html": props["combined_description_soup"].prettify(),
            "summary_html": props["summary_soup"].prettify(),
            "tags": " ".join(tags),
        }
    )
    # traits = []
    # for key in props.keys():
    #     if "has_" in key:
    #         if not props[key]:
    #             traits.append(key)
    # if len(traits) > 0:
    #     print("-" * 40)
    #     print(
    #         props["title_text"],
    #         ", ".join([t.replace("has_", "") for t in traits]),
    #     )
    #     # display(HTML(props["summary_soup"].prettify()))
    #     display(HTML(props["combined_description_soup"].prettify()))
    #     # if props["has_examples"]:
    #     #     display(HTML(props["examples_soup"].prettify()))

print("Final counter:", counter)

Final counter: 491


In [53]:
for entry in entries:
    display(HTML(entry["title_p_html"]))
    display(HTML(entry["summary_html"]))
    display(HTML(entry["description_html"]))

Integer,Enumerated constant,Description
1,KPIValue,The actual value
2,KPIGoal,A target value
3,KPIStatus,The state of the KPI at a specific moment in time
4,KPITrend,A measure of the value over time
5,KPIWeight,A relative importance assigned to the KPI
6,KPICurrentTimeMember,A temporal context for the KPI


Formula,Description
=NOT(A2>100),A2 is NOT greater than 100
"=IF(AND(NOT(A2>1),NOT(A2<100)),A2,""The value is out of range"")","50 is greater than 1 (TRUE), AND 50 is less than 100 (TRUE), so NOT reverses both arguments to FALSE. AND requires both arguments to be TRUE, so it returns the result if FALSE."
"=IF(OR(NOT(A3<0),NOT(A3>50)),A3,""The value is out of range"")","100 is not less than 0 (FALSE), and 100 is greater than 50 (TRUE), so NOT reverses the arguments to TRUE/FALSE. OR only requires one argument to be TRUE, so it returns the result if TRUE."


Data,,
Statements,,
Profit Margin,,
margin,,
"The ""boss"" is here.",,
Formula,Description,Result
"=SEARCH(""e"",A2,6)","Position of the first ""e"" in the string in cell A2, starting at the sixth position.",7
"=SEARCH(A4,A3)","Position of ""margin"" (string for which to search is cell A4) in ""Profit Margin"" (cell in which to search is A3).",8
"=REPLACE(A3,SEARCH(A4,A3),6,""Amount"")","Replaces ""Margin"" with ""Amount"" by first searching for the position of ""Margin"" in cell A3, and then replacing that character and the next five characters with the string ""Amount.""",Profit Amount
"=MID(A3,SEARCH("" "",A3)+1,4)","Returns the first four characters that follow the first space character in ""Profit Margin"" (cell A3).",Marg
"=SEARCH("""""""",A5)","Position of the first double quotation mark ("") in cell A5.",5


In [57]:
df = pd.DataFrame(entries)
df.rename(
    columns={
        "title_p_html": "TitleHtml",
        "description_html": "DescriptionHtml",
        "summary_html": "SummaryHtml",
    },
    inplace=True,
)
df.head()

,TitleHtml,DescriptionHtml,SummaryHtml,tags
0,<p>\n CUBEKPIMEMBER\n</p>\n,<h2>\n CUBEKPIMEMBER\n</h2>\n<section aria-lab...,<p>\n Returns a key performance indicator (KPI...,MicrosoftOffice-2021 ExcelHasExamples ExcelHas...
1,<p>\n ERFC\n</p>\n,"<h2>\n ERFC\n</h2>\n<section aria-labelledby=""...",<p>\n Returns the complementary ERF function i...,MicrosoftOffice-2021 ExcelHasRemarks ExcelHasS...
2,<p>\n LARGE\n</p>\n,<h2>\n LARGE\n</h2>\n<section aria-labelledby=...,<p>\n Returns the k-th largest value in a data...,MicrosoftOffice-2021 ExcelHasRemarks ExcelHasS...
3,<p>\n NOT\n</p>\n,"<h2>\n NOT\n</h2>\n<section class=""ocpIntroduc...","<p>\n Use the ___ function, one of the logical...",MicrosoftOffice-2021 ExcelHasExamples
4,<p>\n BITRSHIFT\n</p>\n,<h2>\n BITRSHIFT\n</h2>\n<section aria-labelle...,<p>\n Returns a number shifted right by the sp...,MicrosoftOffice-2021 ExcelHasRemarks ExcelHasS...


In [63]:
df.sample(n=10)

,TitleHtml,DescriptionHtml,SummaryHtml,tags
30,<p>\n DATEVALUE\n</p>\n,<h2>\n DATEVALUE\n</h2>\n<section aria-labelle...,<p>\n The ___ function converts a date that is...,MicrosoftOffice-2021 ExcelHasRemarks ExcelHasS...
446,<p>\n NORMSDIST\n</p>\n,"<h2>\n NORMSDIST\n</h2>\n<section class=""ocpIn...",<p>\n Returns the standard normal cumulative d...,MicrosoftOffice-2021 ExcelHasRemarks ExcelHasS...
267,<p>\n MINIFS\n</p>\n,"<h2>\n MINIFS\n</h2>\n<section class=""ocpIntro...",<p>\n The ___ function returns the minimum val...,MicrosoftOffice-2021 ExcelHasExamples ExcelHas...
485,<p>\n ASC\n</p>\n,"<h2>\n ASC\n</h2>\n<section aria-labelledby=""I...",<p>\n For Double-byte character set (DBCS) lan...,MicrosoftOffice-2021 ExcelHasSyntax
78,<p>\n SUM\n</p>\n,"<h2>\n SUM\n</h2>\n<section class=""ocpIntroduc...",<p>\n The ___ function adds values.\n</p>\n,MicrosoftOffice-2021
118,<p>\n T.INV\n</p>\n,"<h2>\n T.INV\n</h2>\n<section class=""ocpIntrod...",<p>\n This article describes the formula synta...,MicrosoftOffice-2021 ExcelHasRemarks ExcelHasS...
288,<p>\n AVERAGEA\n</p>\n,<h2>\n AVERAGEA\n</h2>\n<section aria-labelled...,<p>\n Calculates the average (arithmetic mean)...,MicrosoftOffice-2021 ExcelHasRemarks ExcelHasS...
210,<p>\n N\n</p>\n,"<h2>\n N\n</h2>\n<section aria-labelledby=""ID0...",<p>\n Returns a value converted to a number.\n...,MicrosoftOffice-2021 ExcelHasRemarks ExcelHasS...
26,<p>\n COTH\n</p>\n,"<h2>\n COTH\n</h2>\n<section aria-labelledby=""...",<p>\n Return the hyperbolic cotangent of a hyp...,MicrosoftOffice-2021 ExcelHasRemarks ExcelHasS...
69,<p>\n NORMSINV\n</p>\n,"<h2>\n NORMSINV\n</h2>\n<section class=""ocpInt...",<p>\n Returns the inverse of the standard norm...,MicrosoftOffice-2021 ExcelHasRemarks ExcelHasS...


In [64]:
df.to_csv(target_abspath, sep="|", index=False, header=False)